In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/work/data/vessel_catalog.csv'):
    os.system('mkdir -p /content/work/data /content/work/model /content/work/output /content/raw /content/baseline')
    os.system('unzip -o -q /content/drive/MyDrive/vessel/open_data.zip -d /content/raw')
    os.system('unzip -o -q /content/raw/open.zip -d /content/work/data')
    os.system('unzip -o -q /content/raw/baseline_submit.zip -d /content/baseline')
print(os.listdir('/content/work/data'))

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

D = '/content/work/data'
train = pd.read_csv(f'{D}/train.csv')
cat   = pd.read_csv(f'{D}/vessel_catalog.csv')
sub   = pd.read_csv(f'{D}/sample_submission.csv')

print(train.shape, cat.shape, sub.shape)

In [ ]:
print('== train 컬럼')
print(train.dtypes)
print('\n== catalog 컬럼')
print(cat.dtypes)

display(train.head(10))
display(cat.head(10))

In [ ]:
m = train.merge(cat, left_on='target_vessel_id', right_on='vessel_id', suffixes=('_rec', '_cat'))

cols = ['vessel_name_rec', 'vessel_name_cat',
        'gross_tonnage_rec', 'gross_tonnage_cat',
        'flag_name_rec', 'flag_name_cat',
        'vessel_type_rec', 'vessel_type_cat']
print(m[cols].sample(20, random_state=0).to_string())

# 데이터 검증셋 분할(선박 단위,고정)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(train, groups=train['target_vessel_id']))
tr, va = train.iloc[tr_idx], train.iloc[va_idx]

print('학습용:', len(tr), '/ 검증용:', len(va))
print('양쪽에 겹치는 선박 수:', len(set(tr['target_vessel_id']) & set(va['target_vessel_id'])))
print('검증용 중 호출부호로 찾을 수 있는 비율:', round(va['call_sign'].isin(set(tr['call_sign'])).mean(), 3))

is_kor = lambda s: s.astype(str).str.contains('[가-힣]')
print('한글 비율 - 학습용:', round(is_kor(tr['vessel_name']).mean(), 3),
      '/ 검증용:', round(is_kor(va['vessel_name']).mean(), 3))

# 여기 아래부턴 실험 진행 코드

In [ ]:
# 단어 사전 만드는 셀

from collections import Counter, defaultdict

id2name = dict(zip(cat['vessel_id'], cat['vessel_name']))

def ko_tokens(s):
    toks = str(s).split()
    # 마지막 단어 끝의 '호'(~號) 제거: '파이오니어호' -> '파이오니어'
    if toks and len(toks[-1]) > 1 and toks[-1].endswith('호'):
        toks[-1] = toks[-1][:-1]
    return toks

def en_tokens(s):
    return str(s).upper().split()

# 1) 학습용 한글 기록에서 단어 쌍 모으기
kor_tr = tr[is_kor(tr['vessel_name'])]
pair_count = Counter()
n_aligned = 0
for ko, vid in zip(kor_tr['vessel_name'], kor_tr['target_vessel_id']):
    k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
    if len(k) == len(e):
        n_aligned += 1
        pair_count.update(zip(k, e))
print(f'단어 수가 같아서 짝지을 수 있었던 기록: {n_aligned} / {len(kor_tr)}')

# 2) 한글 단어마다 가장 자주 짝지어진 영문 단어 하나를 고름
ko2en = defaultdict(Counter)
for (k, e), c in pair_count.items():
    ko2en[k][e] += c
word_dict = {k: cnt.most_common(1)[0][0] for k, cnt in ko2en.items()}
print('사전 단어 수:', len(word_dict))

print('\n자주 나온 쌍 상위 30개:')
for (k, e), c in pair_count.most_common(30):
    print(f'  {k} -> {e}  ({c}회)')

# 3) 검증용 한글 단어 중 사전에 있는 비율
kor_va = va[is_kor(va['vessel_name'])]
va_toks = [t for s in kor_va['vessel_name'] for t in ko_tokens(s)]
print('\n검증용 한글 단어 중 사전에 있는 비율:', round(sum(t in word_dict for t in va_toks) / len(va_toks), 3))

In [ ]:
!pip install -q rapidfuzz==3.14.3 Unidecode==1.4.0

import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode

def translate(name):
    out = []
    for t in ko_tokens(name):
        out.append(word_dict[t] if t in word_dict else unidecode(t).upper())
    return ' '.join(out)

cat_names = cat['vessel_name'].astype(str).str.upper().values
cat_gt    = cat['gross_tonnage'].values
cat_ids   = cat['vessel_id'].values
all_idx   = np.arange(len(cat))

def match(name, gt, tol=0.10):
    q = translate(name)
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    best = process.extractOne(q, cat_names[idx], scorer=fuzz.ratio)
    return cat_ids[idx[best[2]]], q

kor_va = va[is_kor(va['vessel_name'])].copy()
gt_in = kor_va['gross_tonnage'].fillna(kor_va['international_gross_tonnage'])
res = [match(n, g) for n, g in zip(kor_va['vessel_name'], gt_in)]
kor_va['pred2'] = [r[0] for r in res]
kor_va['변환'] = [r[1] for r in res]

acc = (kor_va['pred2'] == kor_va['target_vessel_id']).mean()
print(f'한글 정확도: 베이스라인 0.374 → 사전 매칭 {acc:.3f}')

kor_va['정답_이름'] = kor_va['target_vessel_id'].map(id2name)
print(kor_va[['vessel_name', '변환', '정답_이름']].head(20).to_string())

In [ ]:
import re

VOW = set('AEIOUY')

# 한글 초성 19개, 종성 28개를 뼈대 자음으로
CHO  = ['K','K','N','T','T','R','M','P','P','S','S','','J','J','J','K','T','P','H']
JONG = ['','K','K','K','N','N','N','T','R','R','R','R','R','R','R','R',
        'M','P','P','T','T','N','T','T','K','T','P','']

def collapse(s):
    # 같은 글자가 연속되면 하나로: 'KSPRR' -> 'KSPR'
    return re.sub(r'(.)\1+', r'\1', s)

def ko_key(word):
    out = []
    for ch in str(word):
        if '가' <= ch <= '힣':
            code = ord(ch) - 0xAC00
            out.append(CHO[code // 588] + JONG[code % 28])
        elif ch.isdigit():
            out.append(ch)
    return collapse(''.join(out))

def en_key(word):
    s = re.sub(r'[^A-Z0-9]', '', str(word).upper())
    out, i, n = [], 0, len(s)
    while i < n:
        c, nx, two = s[i], s[i+1:i+2], s[i:i+2]
        if c.isdigit():                  out.append(c); i += 1
        elif two == 'PH':                out.append('P'); i += 2
        elif two in ('TH', 'SH'):        out.append('S'); i += 2
        elif two == 'CH':                out.append('J'); i += 2
        elif two == 'CK':                out.append('K'); i += 2
        elif two == 'NG' and s[i+2:i+3] not in VOW:
                                         out.append('N'); i += 2
        elif c in VOW or c == 'W':       i += 1
        elif c == 'H':                   out.append('H' if nx in VOW else ''); i += 1
        elif c == 'C':                   out.append('S' if nx and nx in 'EIY' else 'K'); i += 1
        elif c == 'G':                   out.append('J' if nx and nx in 'EI' else 'K'); i += 1
        elif c == 'X':                   out.append('KS'); i += 1
        elif c in 'QK':                  out.append('K'); i += 1
        elif c in 'BPVF':                out.append('P'); i += 1
        elif c in 'DT':                  out.append('T'); i += 1
        elif c in 'LR':                  out.append('R'); i += 1
        elif c in 'ZJ':                  out.append('J'); i += 1
        else:                            out.append(c); i += 1
    return collapse(''.join(out))

def query_key(name):
    # 사전에 있는 단어는 영문으로 바꾼 뒤 영문 규칙, 없는 단어는 한글 규칙
    parts = []
    for t in ko_tokens(name):
        parts.append(en_key(word_dict[t]) if t in word_dict else ko_key(t))
    return ''.join(parts)

for ko, en in [('네스토르', 'NESTOR'), ('헌터', 'HUNTER'), ('머큐리', 'MERCURY'),
               ('갤럭시', 'GALAXY'), ('익스플러어', 'EXPLORER'), ('제니스', 'JANICE')]:
    print(f'{ko:6s} {ko_key(ko):8s} | {en:9s} {en_key(en)}')

In [ ]:
import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode

def translate(name):
    return ' '.join(word_dict.get(t, unidecode(t).upper()) for t in ko_tokens(name))

cat_names = cat['vessel_name'].astype(str).str.upper().values
cat_keys  = np.array([en_key(n) for n in cat['vessel_name']])
cat_gt    = cat['gross_tonnage'].values
cat_ids   = cat['vessel_id'].values
all_idx   = np.arange(len(cat))

def predict_variant(name, gt, mode, tol=0.10):
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    s_text = process.cdist([translate(name)], cat_names[idx], scorer=fuzz.ratio)[0]
    s_key  = process.cdist([query_key(name)], cat_keys[idx], scorer=fuzz.ratio)[0]
    score = {'text': s_text, 'key': s_key, 'sum': s_text + s_key}[mode]
    return cat_ids[idx[np.argmax(score)]]

kor_va = va[is_kor(va['vessel_name'])].copy()
gt_intl = kor_va['international_gross_tonnage'].fillna(kor_va['gross_tonnage'])

for mode in ['text', 'key', 'sum']:
    p = [predict_variant(n, g, mode) for n, g in zip(kor_va['vessel_name'], gt_intl)]
    kor_va[f'pred_{mode}'] = p
    acc = (np.array(p) == kor_va['target_vessel_id'].values).mean()
    print(f'{mode:5s}: {acc:.3f}')

In [ ]:
LETTERS = {'더블유':'W','에이치':'H','에이':'A','에프':'F','에스':'S','엑스':'X','제트':'Z',
           '제이':'J','케이':'K','브이':'V','와이':'Y','아이':'I','엘':'L','엠':'M','엔':'N',
           '앤':'N','알':'R','비':'B','씨':'C','시':'C','디':'D','이':'E','지':'G','오':'O',
           '피':'P','큐':'Q','티':'T','유':'U'}
L_KEYS = sorted(LETTERS, key=len, reverse=True)
NUM_WORD = {'투': 'II', '쓰리': 'III'}

def parse_letters(t):
    out, i = [], 0
    while i < len(t):
        for k in L_KEYS:
            if t.startswith(k, i):
                out.append(LETTERS[k]); i += len(k); break
        else:
            return None
    return ''.join(out) if len(out) >= 2 else None

def expand_tokens(name):
    out = []
    for t in ko_tokens(name):
        m = re.match(r'^제(\d+)(.*)$', t)
        if m:
            out.append('NO.' + m.group(1))
            if m.group(2):
                out.append(m.group(2))
        else:
            out.append(t)
    return out

def conv(t):
    if t in word_dict:             return word_dict[t], True
    if not re.search('[가-힣]', t):  return t.upper(), True
    if t in NUM_WORD:              return NUM_WORD[t], True
    L = parse_letters(t)
    if L:                          return L, True
    return unidecode(t).upper(), False

def translate2(name):
    return ' '.join(conv(t)[0] for t in expand_tokens(name))

def query_key2(name):
    parts = []
    for t in expand_tokens(name):
        text, is_en = conv(t)
        parts.append(en_key(text) if is_en else ko_key(t))
    return ''.join(parts)

def predict2(name, gt, tol=0.10):
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    s_text = process.cdist([translate2(name)], cat_names[idx], scorer=fuzz.ratio)[0]
    s_key  = process.cdist([query_key2(name)], cat_keys[idx], scorer=fuzz.ratio)[0]
    return cat_ids[idx[np.argmax(s_text + s_key)]]

kor_va['pred_v2'] = [predict2(n, g) for n, g in zip(kor_va['vessel_name'], gt_intl)]
print(f"한글 정확도: {(kor_va['pred_v2'] == kor_va['target_vessel_id']).mean():.3f}")

In [ ]:
# 1) train의 톤수 결측 패턴
im = train['international_gross_tonnage'].isna()
dm = train['gross_tonnage'].isna()
k_all = is_kor(train['vessel_name'])
for label, mask in [('한글', k_all), ('비한글', ~k_all)]:
    print(f'[{label}] 국제만 없음: {(im & ~dm & mask).sum()} / 국내만 없음: {(~im & dm & mask).sum()} '
          f'/ 둘 다 없음: {(im & dm & mask).sum()} / 전체: {mask.sum()}')

# 2) 비한글: 국내 총톤수만 줬을 때 범위별 정확도
def predict_range(name, lo, hi):
    idx = all_idx
    if lo is not None:
        cand = np.where((cat_gt >= lo) & (cat_gt <= hi))[0]
        if len(cand) > 0:
            idx = cand
    best = process.extractOne(str(name).upper(), cat_names[idx], scorer=fuzz.ratio)
    return cat_ids[idx[best[2]]]

eng = va[~is_kor(va['vessel_name']) & va['gross_tonnage'].notna()]
print(f'\n비한글 검증 기록 {len(eng)}건, 국내 총톤수만 사용')
for lo_r, hi_r in [(0.9, 1.1), (0.9, 1.6)]:
    pred = [predict_range(n, g * lo_r, g * hi_r) for n, g in zip(eng['vessel_name'], eng['gross_tonnage'])]
    print(f'  × {lo_r} ~ {hi_r}: {(np.array(pred) == eng["target_vessel_id"].values).mean():.3f}')

#submit 파일 생성


In [ ]:
import json

!rm -rf /content/submit_v8b && mkdir -p /content/submit_v8b/model

kor_all = train[is_kor(train['vessel_name'])]
pc = Counter()
for ko, vid in zip(kor_all['vessel_name'], kor_all['target_vessel_id']):
    k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
    if len(k) == len(e):
        pc.update(zip(k, e))

k2e = defaultdict(Counter)
for (k, e), c in pc.items():
    k2e[k][e] += c
word_dict_full = {k: cnt.most_common(1)[0][0] for k, cnt in k2e.items()}

with open('/content/submit_v8b/model/word_dict.json', 'w', encoding='utf-8') as f:
    json.dump(word_dict_full, f, ensure_ascii=False)

with open('/content/submit_v8b/requirements.txt', 'w') as f:
    f.write('# Uses the evaluation image\'s preinstalled numpy, pandas, rapidfuzz and Unidecode.\n')

print('전체 사전 단어 수:', len(word_dict_full))

In [ ]:
%%writefile /content/submit_v8b/model/matcher.py
"""Vessel matcher.

Korean names: token rules (No.N, alphabet acronyms, number words) + dictionary
  -> text similarity + consonant-skeleton similarity
Non-Korean names: text similarity
Both: international gross tonnage filter (+-10%).
"""

import json
import re

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode

HANGUL = re.compile("[가-힣]")
VOW = set("AEIOUY")
CHO = ["K","K","N","T","T","R","M","P","P","S","S","","J","J","J","K","T","P","H"]
JONG = ["","K","K","K","N","N","N","T","R","R","R","R","R","R","R","R",
        "M","P","P","T","T","N","T","T","K","T","P",""]
LETTERS = {"더블유":"W","에이치":"H","에이":"A","에프":"F","에스":"S","엑스":"X","제트":"Z",
           "제이":"J","케이":"K","브이":"V","와이":"Y","아이":"I","엘":"L","엠":"M","엔":"N",
           "앤":"N","알":"R","비":"B","씨":"C","시":"C","디":"D","이":"E","지":"G","오":"O",
           "피":"P","큐":"Q","티":"T","유":"U"}
L_KEYS = sorted(LETTERS, key=len, reverse=True)
NUM_WORD = {"투": "II", "쓰리": "III"}


def is_korean(name):
    return bool(HANGUL.search(str(name)))


def ko_tokens(s):
    toks = str(s).split()
    if toks and len(toks[-1]) > 1 and toks[-1].endswith("호"):
        toks[-1] = toks[-1][:-1]
    return toks


def expand_tokens(name):
    out = []
    for t in ko_tokens(name):
        m = re.match(r"^제(\d+)(.*)$", t)
        if m:
            out.append("NO." + m.group(1))
            if m.group(2):
                out.append(m.group(2))
        else:
            out.append(t)
    return out


def parse_letters(t):
    out, i = [], 0
    while i < len(t):
        for k in L_KEYS:
            if t.startswith(k, i):
                out.append(LETTERS[k]); i += len(k); break
        else:
            return None
    return "".join(out) if len(out) >= 2 else None


def collapse(s):
    return re.sub(r"(.)\1+", r"\1", s)


def ko_key(word):
    out = []
    for ch in str(word):
        if "가" <= ch <= "힣":
            code = ord(ch) - 0xAC00
            out.append(CHO[code // 588] + JONG[code % 28])
        elif ch.isdigit():
            out.append(ch)
    return collapse("".join(out))


def en_key(word):
    s = re.sub(r"[^A-Z0-9]", "", str(word).upper())
    out, i, n = [], 0, len(s)
    while i < n:
        c, nx, two = s[i], s[i+1:i+2], s[i:i+2]
        if c.isdigit():                  out.append(c); i += 1
        elif two == "PH":                out.append("P"); i += 2
        elif two in ("TH", "SH"):        out.append("S"); i += 2
        elif two == "CH":                out.append("J"); i += 2
        elif two == "CK":                out.append("K"); i += 2
        elif two == "NG" and s[i+2:i+3] not in VOW:
                                         out.append("N"); i += 2
        elif c in VOW or c == "W":       i += 1
        elif c == "H":                   out.append("H" if nx in VOW else ""); i += 1
        elif c == "C":                   out.append("S" if nx and nx in "EIY" else "K"); i += 1
        elif c == "G":                   out.append("J" if nx and nx in "EI" else "K"); i += 1
        elif c == "X":                   out.append("KS"); i += 1
        elif c in "QK":                  out.append("K"); i += 1
        elif c in "BPVF":                out.append("P"); i += 1
        elif c in "DT":                  out.append("T"); i += 1
        elif c in "LR":                  out.append("R"); i += 1
        elif c in "ZJ":                  out.append("J"); i += 1
        else:                            out.append(c); i += 1
    return collapse("".join(out))


class VesselMatcher:
    def __init__(self, catalog, dict_path, tol=0.10):
        with open(dict_path, encoding="utf-8") as f:
            self.word_dict = json.load(f)
        self.names = catalog["vessel_name"].astype(str).str.upper().values
        self.keys = np.array([en_key(n) for n in catalog["vessel_name"]])
        self.gt = catalog["gross_tonnage"].values
        self.ids = catalog["vessel_id"].values
        self.all_idx = np.arange(len(catalog))
        self.tol = tol

    def conv(self, t):
        if t in self.word_dict:
            return self.word_dict[t], True
        if not HANGUL.search(t):
            return t.upper(), True
        if t in NUM_WORD:
            return NUM_WORD[t], True
        letters = parse_letters(t)
        if letters:
            return letters, True
        return unidecode(t).upper(), False

    def translate(self, name):
        return " ".join(self.conv(t)[0] for t in expand_tokens(name))

    def query_key(self, name):
        parts = []
        for t in expand_tokens(name):
            text, is_en = self.conv(t)
            parts.append(en_key(text) if is_en else ko_key(t))
        return "".join(parts)

    def predict_one(self, name, gt_intl, gt_dom):
        # 국제총톤수: +-10% / 국내 총톤수만: 0.9~1.6배 (국내 <= 국제) / 둘 다 없음: 전체
        idx = self.all_idx
        if pd.notna(gt_intl):
            lo, hi = gt_intl * (1 - self.tol), gt_intl * (1 + self.tol)
        elif pd.notna(gt_dom):
            if is_korean(name):
              lo, hi = gt_dom * 0.9, gt_dom * 1.6
            else:
              lo, hi = gt_dom * (1 - self.tol),gt_dom*(1+self.tol)
        else:
            lo = hi = None
        if lo is not None:
            cand = np.where((self.gt >= lo) & (self.gt <= hi))[0]
            if len(cand) > 0:
                idx = cand

        if not is_korean(name):
            best = process.extractOne(str(name).upper(), self.names[idx], scorer=fuzz.ratio)
            return self.ids[idx[best[2]]]

        s_text = process.cdist([self.translate(name)], self.names[idx], scorer=fuzz.ratio)[0]
        s_key = process.cdist([self.query_key(name)], self.keys[idx], scorer=fuzz.ratio)[0]
        return self.ids[idx[np.argmax(s_text + s_key)]]

In [ ]:
%%writefile /content/submit_v8b/script.py
"""Vessel matching inference: dictionary + romanization + consonant skeleton + tonnage filter."""

from pathlib import Path
import sys

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent / "model"))
from matcher import VesselMatcher


def run(
    test_path="data/test.csv",
    catalog_path="data/vessel_catalog.csv",
    dict_path="model/word_dict.json",
    output_path="output/submission.csv",
):
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    catalog = pd.read_csv(catalog_path, encoding="utf-8-sig")
    if test["sample_id"].isna().any() or not test["sample_id"].is_unique:
        raise ValueError("test.csv sample_id must be non-null and unique")

    matcher = VesselMatcher(catalog, dict_path)

    # 카탈로그와 같은 국제총톤수를 우선 사용
        # 국제총톤수와 국내 총톤수를 따로 넘겨, 있는 정보에 따라 후보 범위를 다르게 설정
    predictions = [
        matcher.predict_one(n, gi, gd)
        for n, gi, gd in zip(test["vessel_name"], test["international_gross_tonnage"], test["gross_tonnage"])
    ]

    if len(predictions) != len(test) or not set(predictions).issubset(set(catalog["vessel_id"])):
        raise ValueError("Invalid catalog predictions")
    result = pd.DataFrame({"sample_id": test["sample_id"], "target_vessel_id": predictions})
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved {len(result)} predictions to {output_path}")
    return output_path


if __name__ == "__main__":
    run()

In [ ]:
!rm -rf /content/test_run && cp -r /content/submit_v8b /content/test_run
!mkdir -p /content/test_run/data
!cp /content/work/data/vessel_catalog.csv /content/test_run/data/
va.drop(columns=['target_vessel_id']).to_csv('/content/test_run/data/test.csv', index=False)

!cd /content/test_run && time python script.py
!head -5 /content/test_run/output/submission.csv

In [ ]:
!cd /content/submit_v8b && rm -f /content/submit_8b.zip && zip -r /content/submit_8b.zip script.py requirements.txt model
!unzip -l /content/submit_8b.zip

from google.colab import files
files.download('/content/submit_8b.zip')